# 대법원 캡차 이미지 수집

`https://ssgo.scourt.go.kr/ssgo/ssgo10l/getCaptchaInf.on` API를 호출하여 캡차 이미지를 다운로드합니다.


In [4]:
# 필요한 라이브러리 임포트
import requests
from pathlib import Path
from datetime import datetime
import time
from PIL import Image
from io import BytesIO

url = "https://ssgo.scourt.go.kr/ssgo/ssgo10l/getCaptchaInf.on"

In [5]:
# 저장 경로 설정
base_path = Path("captcha_data/supreme_court/0/images/draft")
base_path.mkdir(parents=True, exist_ok=True)

print(f"저장 경로: {base_path.absolute()}")

저장 경로: /home/hyper/project/hyper-captcha-resolver/captcha_data/supreme_court/0/images/draft


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['GLOG_minloglevel'] = '2'

import logging, warnings
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

import captchaResolver.engine as engine
from captchaResolver.dataclass import TrainData
from captchaResolver.core import PyTorchModel

captcha_id = 'supreme_court'
rev = 0
backend = 'pytorch'

model: PyTorchModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
train_data: TrainData = model.train_data
train_data.rev = rev
# image_width: int = 200
# image_height: int = 50
engine.batch_predict_model(model=model)
model_path = train_data.get_model_path()
image_path = train_data.choice_pred_image()
pred, confidence = engine.predict(model=model, image_path=image_path)
print("image_path : ", image_path)
print("pred : ", pred)
print("confidence : ", f'{confidence:.4f}')
print("Done!")


In [6]:
def download_captcha(save_path: Path, index: int, total: int) -> bool:
    """
    캡차 이미지를 다운로드하고 저장합니다.
    """
    import base64
    # 브라우저에서 호출한 것처럼 보이게 할 헤더
    headers = {
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'ko,en;q=0.9,en-US;q=0.8',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36 Edg/142.0.0.0',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        image_b64 = None
        if isinstance(data, dict):
            if 'data' in data and isinstance(data['data'], dict):
                dma = data['data'].get('dma_captchaInf') or data['data'].get('dmaCaptchaInf')
                if isinstance(dma, dict):
                    image_b64 = dma.get('image')
            if not image_b64:
                image_b64 = data.get('image') or data.get('captchaImage')

        if not image_b64:
            raise ValueError('JSON 응답에 이미지(base64) 필드가 없습니다')

        image_bytes = base64.b64decode(image_b64)

        # 파일명/경로 준비 및 저장
        save_path.mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        filename = f"{timestamp}.png"
        filepath = save_path / filename
        with open(filepath, "wb") as f:
            f.write(image_bytes)
        # image.save(filepath, format="PNG")
        print(f"[{index + 1}/{total}] 저장 완료: {filename}")
        return True
    except Exception as e:
        print(f"[{index + 1}/{total}] 오류 발생: {e}")
        return False

In [10]:
# 다운로드 설정
TARGET_COUNT = 500  # 다운로드할 이미지 개수
DELAY = 0.25         # 요청 간 대기 시간 (초)

print(f"캡차 이미지 {TARGET_COUNT}개 다운로드 시작...")
print(f"요청 간격: {DELAY}초")
print("-" * 50)

캡차 이미지 500개 다운로드 시작...
요청 간격: 0.25초
--------------------------------------------------


In [11]:
# 이미지 다운로드 실행
success_count = 0
fail_count = 0

for i in range(TARGET_COUNT):
    if download_captcha(base_path, i, TARGET_COUNT):
        success_count += 1
    else:
        fail_count += 1
    
    # 마지막 요청이 아니면 대기
    if i < TARGET_COUNT - 1:
        time.sleep(DELAY)

print("-" * 50)
print(f"\n다운로드 완료!")
print(f"성공: {success_count}개")
print(f"실패: {fail_count}개")
print(f"저장 위치: {base_path.absolute()}")

[1/500] 저장 완료: 20251114_165519_872474.png
[2/500] 저장 완료: 20251114_165520_871387.png
[3/500] 저장 완료: 20251114_165521_400237.png
[4/500] 저장 완료: 20251114_165521_911119.png
[5/500] 저장 완료: 20251114_165522_418293.png
[6/500] 저장 완료: 20251114_165523_157239.png
[7/500] 저장 완료: 20251114_165523_667226.png
[8/500] 저장 완료: 20251114_165524_118899.png
[9/500] 저장 완료: 20251114_165524_644872.png
[10/500] 저장 완료: 20251114_165525_123085.png
[11/500] 저장 완료: 20251114_165525_576118.png
[12/500] 저장 완료: 20251114_165526_019990.png
[13/500] 저장 완료: 20251114_165526_515095.png
[14/500] 저장 완료: 20251114_165527_041432.png
[15/500] 저장 완료: 20251114_165527_541428.png
[16/500] 저장 완료: 20251114_165528_000181.png
[17/500] 저장 완료: 20251114_165528_680578.png
[18/500] 저장 완료: 20251114_165529_146428.png
[19/500] 저장 완료: 20251114_165529_708516.png
[20/500] 저장 완료: 20251114_165530_260590.png
[21/500] 저장 완료: 20251114_165530_703866.png
[22/500] 저장 완료: 20251114_165531_202971.png
[23/500] 저장 완료: 20251114_165531_674202.png
[24/500] 저장 완료: 2025

In [14]:
def bg_white(source_dir, target_dir):
    # 이미지 투명배경 흰색으로
    from pathlib import Path
    from PIL import Image

    # 경로 설정
    source_dir = Path("captcha_data/supreme_court/0/images/draft")
    target_dir = Path("captcha_data/supreme_court/0/images/draft-white-bg")
    target_dir.mkdir(parents=True, exist_ok=True)

    # 이미지 파일 목록
    image_files = list(source_dir.glob("*.png"))
    print(f"처리할 이미지 개수: {len(image_files)}")

    # 각 이미지 처리: 투명 배경 -> 흰색 배경
    success_count = 0
    for idx, img_path in enumerate(image_files, 1):
        try:
            img = Image.open(img_path)
            if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
                # img_rgba = img.convert("RGB")
                bg = Image.new("RGB", img.size, (255, 255, 255))
                alpha = img.split()[-1]
                bg.paste(img, mask=alpha)
                img_final = bg
                img = img_final.convert("RGB")
            
            target_path = target_dir / img_path.name
            img.save(target_path, format="PNG")

            success_count += 1
            if idx % 50 == 0 or idx == len(image_files):
                print(f"[{idx}/{len(image_files)}] 처리 완료")

        except Exception as e:
            print(f"[{idx}/{len(image_files)}] 오류 발생 ({img_path.name}): {e}")

    print(f"\n작업 완료! 성공: {success_count}/{len(image_files)}")
    print(f"저장 위치: {target_dir.absolute()}")
    
    

### 캡차 이미지 인식 및 파일명 변경

학습된 모델을 사용하여 draft 폴더의 이미지를 인식하고, 예측된 레이블로 파일명을 변경합니다.

In [ ]:
import os
from pathlib import Path
from captchaResolver.dataclass import TrainData
import captchaResolver.engine as engine
from captchaResolver.core import PyTorchModel

captcha_id = 'supreme_court'
backend = 'pytorch'
rev = 0
draft_image_dir = "captcha_data/supreme_court/0/images/draft"
labled_image_dir = "captcha_data/supreme_court/0/images/labled"

Path(labled_image_dir).mkdir(parents=True, exist_ok=True)

draft_image_files = sorted([str(p) for p in Path(draft_image_dir).glob("*.png")])
print(f"Draft 이미지 개수: {len(draft_image_files)}")

model: PyTorchModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
train_data: TrainData = model.train_data
model.train_data.rev = rev
torch_model: PyTorchModel = model
matched = 0
# pred_img_path_list = torch_model.train_data.get_data_files(train=False)
# pred_dataset = tf.data.Dataset.from_tensor_slices((pred_img_path_list))
# pred_dataset = (
#     pred_dataset
#     .map(keras_model.encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
#     .batch(batch_size)
#     .prefetch(buffer_size=tf.data.AUTOTUNE)
# )

# Load prediction model if not loaded
torch_model.load_prediction_model()

for idx, img_path in enumerate(draft_image_files):
    pred, confidence = engine.predict(model=model, image_path=img_path, verbose=0)
    # rename draft image with predicted text
    new_image_path = os.path.join(labled_image_dir, pred + ".png")
    if(os.path.exists(new_image_path)):
        continue
    Path(img_path).rename(new_image_path)
    print(f"[{idx + 1}/{len(draft_image_files)}] image_path : {img_path}")
    print(f"pred : {pred}")
    print(f"confidence : {confidence:.4f}")
    print("new_image_path : ", new_image_path)
    print("Done!")


In [ ]:
# target_dir = Path("captcha_data/gov24/0/images/resize")
import glob


image_list = glob.glob(os.path.join(target_dir, "*.png"))
print(f"리사이즈된 이미지 개수: {len(image_list)}")

In [ ]:
import os, glob, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['GLOG_minloglevel'] = '2'

import logging, warnings
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

import keras
import tensorflow as tf

from pathlib import Path
from captchaResolver.dataclass import TrainData
import captchaResolver.engine as engine
from captchaResolver.keras_core import KerasModel

draft_dir = Path("captcha_data/gov24/1/images/draft")
target_dir = Path("captcha_data/gov24/1/images/labeled")
pred_dir = Path("captcha_data/gov24/1/images/pred")
train_dir = Path("captcha_data/gov24/1/images/train")
captcha_id = 'gov24'
backend = 'keras'
rev = 1
image_width = 200
image_height = 50
batch_size = 32

start = time.time()

model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)
train_data: TrainData = model.train_data
model.train_data.rev = rev
model.train_data.image_width = image_width
model.train_data.image_height = image_height

keras_model: KerasModel = model
matched = 0

pred_img_path_list = sorted(glob.glob(os.path.join(target_dir, "*.png")))
pred_labels = [os.path.basename(p).split(".")[0] for p in pred_img_path_list]
pred_dataset = tf.data.Dataset.from_tensor_slices((pred_img_path_list, pred_labels))
pred_dataset = (
    pred_dataset
    .map(keras_model.encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

# Load prediction model if not loaded
keras_model.load_prediction_model()

# Batch prediction
all_preds = []
all_labels = []

for batch in pred_dataset:
    images = batch["image"]
    labels = batch["label"]
    
    # Predict batch
    pred_vals = keras_model.predict_model.predict(images, verbose=0)
    preds = keras_model.decode_batch_predictions(pred_vals)
    
    # Decode original labels
    for label in labels:
        label_text = tf.strings.reduce_join(
            keras_model.num_to_char(label + 1)
        ).numpy().decode("utf-8")
        all_labels.append(label_text)
    
    all_preds.extend(preds)

# Compare predictions with original labels
for idx, (ori, pred) in enumerate(zip(all_labels, all_preds)):
    img_path = pred_img_path_list[idx]
    msg = ""
    if ori == pred:
        train_img_path = train_dir / f"{ori}.png"
        tf.io.gfile.copy(img_path, train_img_path, overwrite=True)
        matched += 1
    else:
        # 불일치 파일은 pred_dir로 복사
        pred_img_path = pred_dir / f"{ori}.png"
        tf.io.gfile.copy(img_path, pred_img_path, overwrite=True)
        msg = " Not matched!"
    
    # Calculate confidence for display (optional)
    print(f"ori: {ori}, pred: {pred}{msg}")

end = time.time()
total = len(pred_img_path_list)
accuracy = matched / total * 100 if total > 0 else 0

print(f"Matched: {matched}, Total: {total}, Accuracy: {accuracy:.2f}%")
print(f"pred time: {end - start:.2f} sec") 


# engine.batch_predict_model(model=model)
# model_path = train_data.get_model_path()
# image_path = train_data.choice_pred_image()
# pred, confidence = engine.predict(model=model, image_path=image_path)
# print("image_path : ", image_path)
# print("pred : ", pred)
# print("confidence : ", f'{confidence:.4f}')
# print("Done!")


In [ ]:
# captcha_data/gov24/1/images/pred 폴더의 이미지 크기 검사
from pathlib import Path
from PIL import Image

# 경로 설정
pred_dir = Path("captcha_data/gov24/1/images/pred")

if not pred_dir.exists():
    print(f"폴더가 존재하지 않습니다: {pred_dir}")
else:
    # 이미지 파일 목록
    image_files = sorted(pred_dir.glob("*.png"))
    print(f"검사할 이미지 개수: {len(image_files)}")
    print("=" * 70)
    
    # 크기가 다른 이미지 리스트
    wrong_size_images = []
    target_width = 200
    target_height = 50
    
    for img_path in image_files:
        try:
            img = Image.open(img_path)
            width, height = img.size
            
            if width != target_width or height != target_height:
                wrong_size_images.append({
                    'name': img_path.name,
                    'size': f"{width}x{height}"
                })
                
        except Exception as e:
            print(f"⚠️  오류 ({img_path.name}): {e}")
    
    # 결과 출력
    if wrong_size_images:
        print(f"\n❌ 크기가 {target_width}x{target_height}이 아닌 이미지 ({len(wrong_size_images)}개):\n")
        for idx, img_info in enumerate(wrong_size_images, 1):
            print(f"  {idx:3d}. {img_info['name']:30s} -> {img_info['size']}")
    else:
        print(f"\n✅ 모든 이미지가 {target_width}x{target_height} 크기입니다.")
    
    print("=" * 70)
    print(f"\n검사 완료!")
    print(f"전체: {len(image_files)}개")
    print(f"정상 ({target_width}x{target_height}): {len(image_files) - len(wrong_size_images)}개")
    print(f"비정상: {len(wrong_size_images)}개")

In [ ]:
# captcha_data/gov24/1/images/pred 폴더의 파일명 길이 검사 (확장자 포함 10자리가 아닌 것)
from pathlib import Path

# 경로 설정
pred_dir = Path("captcha_data/gov24/0/images/pred")

if not pred_dir.exists():
    print(f"폴더가 존재하지 않습니다: {pred_dir}")
else:
    # 이미지 파일 목록
    image_files = sorted(pred_dir.glob("*.png"))
    print(f"검사할 이미지 개수: {len(image_files)}")
    print("=" * 70)
    
    # 파일명 길이가 10자리(확장자 포함)가 아닌 이미지 리스트
    wrong_name_images = []
    target_length = 10  # 예: "abc12.png" = 9자 (레이블 5자 + ".png" 4자)
    
    for img_path in image_files:
        filename = img_path.name
        name_length = len(filename)
        
        if name_length != target_length:
            wrong_name_images.append({
                'name': filename,
                'length': name_length,
                'label_length': len(img_path.stem)  # 확장자 제외한 레이블 길이
            })
    
    # 결과 출력
    if wrong_name_images:
        print(f"\n❌ 파일명 길이가 {target_length}자가 아닌 이미지 ({len(wrong_name_images)}개):\n")
        for idx, img_info in enumerate(wrong_name_images, 1):
            print(f"  {idx:3d}. {img_info['name']:40s} (길이: {img_info['length']:2d}, 레이블: {img_info['label_length']:2d}자)")
    else:
        print(f"\n✅ 모든 이미지 파일명이 {target_length}자입니다.")
    
    print("=" * 70)
    print(f"\n검사 완료!")
    print(f"전체: {len(image_files)}개")
    print(f"정상 (파일명 {target_length}자): {len(image_files) - len(wrong_name_images)}개")
    print(f"비정상: {len(wrong_name_images)}개")

In [ ]:
pred_dir = Path("captcha_data/gov24/1/images/pred")
train_dir = Path("captcha_data/gov24/1/images/train")
captcha_id = 'gov24'
backend = 'keras'
rev = 1
image_width = 200
image_height = 50
batch_size = 32
model: KerasModel = engine.get_captcha_model(captcha_id=captcha_id, backend=backend)


#### 학습 데이타 섞기

In [ ]:
import captchaResolver.engine as engine

engine.redistribute_train_pred(
    image_dir="captcha_data/gov24/1/images",
    train_ratio=0.9,
    verbose=True,
)